# Job Scout — Phase 3: Ollie reads the traces, then fixes the code

**Build → Evaluate → Self-Improve.** Phase 1 built the agent, Phase 2 measured it. This is
the self-improving part, and it runs on **Ollie** — Opik's assistant, which reads your
traces in the dashboard and, once connected to your repository, reads the code behind them,
proposes an edit, reruns your agent and runs a test suite to show the regression is gone.

Five capabilities, and this notebook walks all of them:

1. **Trace investigation** — span trees, root causes, failing runs versus passing ones
2. **Source-code integration** — reads your files, proposes edits, needs your approval
3. **Agent re-execution** — reruns your agent on the original trace inputs
4. **Test-suite integration** — proves the regression is gone
5. **Cross-workspace search** — traces, datasets, experiments and prompts in one conversation

There is no SDK for any of it, so this notebook's job is to *manufacture the evidence and
hand you the wheel*: it produces the traces, prints the numbers you should expect back, and
gives you the exact questions to ask.

**One real bug carries the whole chapter.** JSearch spends its full 15-second timeout and
returns nothing, on every search — found by the per-source spans Phase 3 added, and left
unfixed on purpose so the loop above has something genuine to fix.

**The lesson, stated up front:** an assistant can only find what your instrumentation
records. The same question, asked before and after those spans existed, gets a different
answer — and only one of them is useful.

Companion notes: [`phase3/README.md`](phase3/README.md) ·
[`../docs/ollie.md`](../docs/ollie.md) · [`../docs/optimizing_latency.md`](../docs/optimizing_latency.md)

## 1. Environment check

This notebook needs an **Opik key** — without traces there is nothing for Ollie to read.
Job source keys are optional: the cascade falls back to the committed cache.

In [ ]:
# The cold-import cycle: import schemas before anything pulls in the graph.
from job_scout.graph.schemas import Profile  # noqa: F401

from job_scout.config import get_settings

settings = get_settings()
checks = {
    "Opik tracing (required here)": settings.has_opik,
    "JSearch key": bool(settings.jsearch_api_key.get_secret_value()),
    "Adzuna keys": bool(settings.adzuna_app_id and settings.adzuna_app_key.get_secret_value()),
    "Concurrent fan-out (SCOUT_CONCURRENT_SOURCES)": settings.scout_concurrent_sources,
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else '-- '} {name}")

if not settings.has_opik:
    print("\nNo Opik key: the searches below still run, but nothing is traced and Ollie has nothing to read.")

## 2. What the spans look like now

Phase 3 wraps each live source in its own span. The wrapper is deliberately a **no-op when
Opik is not configured** — `opik.track` otherwise ships spans without a key and answers 401
into your terminal, and this repo promises to stay quiet when keyless.

Note what is spanned: the **query**, not the consumption. The cascade still decides which
results get merged (the `<5` and `<3` thresholds); the spans just make the waiting visible.

In [ ]:
import inspect

from job_scout.tracing import traced_call

print(inspect.getsource(traced_call))

## 3. Time each source on its own

Before the traces, the ground truth: hit each adapter directly and time it. This is the
number Ollie should be able to find for you afterwards — if it cannot, the instrumentation
is at fault, not the assistant.

In [ ]:
import time

from job_scout.tools.jobs_api import AdzunaSource, JSearchSource, RemotiveSource

QUERY, LOCATION, COUNTRY = "data scientist", "Berlin, Germany", "de"

for name, source in [("jsearch", JSearchSource()), ("adzuna", AdzunaSource()), ("remotive", RemotiveSource())]:
    if not getattr(source, "available", True):
        print(f"{name:10s} (no key — skipped)")
        continue
    started = time.monotonic()
    try:
        found = source.fetch(QUERY, LOCATION, COUNTRY, False, 5)
        print(f"{name:10s} {round((time.monotonic() - started) * 1000):6d} ms   {len(found)} jobs")
    except Exception as exc:  # a dead source is an empty source
        print(f"{name:10s} {round((time.monotonic() - started) * 1000):6d} ms   {type(exc).__name__}")

**What we saw on 2026-08-04** (yours will differ — that is the point of measuring):

```
jsearch     15264 ms   0 jobs
adzuna        892 ms   5 jobs
remotive      157 ms   5 jobs
```

15.0s is exactly `JSearchSource`'s timeout. The **primary** source is not slow — it is
timing out, and contributing nothing while it does. Hold that number.

## 4. Two traces: concurrent, then sequential

The same search twice, so you have a pair to compare in the dashboard. Concurrent fan-out
(the Phase 3 default) makes wall time the **slowest single source**; sequential makes it the
**sum**. Both are traced, both get a link printed below.

In [ ]:
import opik

from job_scout.tools.jobs_api import run_search
from job_scout.tracing import configure_opik

configure_opik()


@opik.track(name="ollie-demo-search")
def traced_search(mode: str) -> dict:
    """One search, traced, tagged with the fan-out mode it ran under."""
    started = time.monotonic()
    jobs, used = run_search(QUERY, LOCATION, COUNTRY, False, 10)
    return {"mode": mode, "ms": round((time.monotonic() - started) * 1000), "jobs": len(jobs), "sources_used": used}


results = []
for concurrent in (True, False):
    get_settings.cache_clear()
    import os

    os.environ["SCOUT_CONCURRENT_SOURCES"] = "true" if concurrent else "false"
    results.append(traced_search("concurrent" if concurrent else "sequential"))
    print(results[-1])

opik.flush_tracker()
print("\nflushed — the traces are in the dashboard now")

## 5. Read your own span tree

Before opening the UI, pull the spans back down. Seeing the tree in code first means you
know the right answer when you go on to ask Ollie for it.

In [ ]:
client = opik.Opik()
project = get_settings().opik_project_name
traces = client.search_traces(project_name=project, filter_string='name = "ollie-demo-search"', max_results=2)

for trace in traces:
    total = round((trace.end_time - trace.start_time).total_seconds() * 1000) if trace.end_time else None
    print(f"\ntrace {trace.id}  total {total} ms  output={trace.output}")
    spans = client.search_spans(project_name=project, trace_id=trace.id, max_results=50)
    for span in sorted(spans, key=lambda s: s.start_time):
        ms = round((span.end_time - span.start_time).total_seconds() * 1000) if span.end_time else None
        print(f"    {span.name:22s} {ms:6} ms")

**The pair we measured on 2026-08-04**, straight out of the cell above:

```
concurrent   15368 ms    source.jsearch 15355 · source.adzuna 914 · source.remotive 218
sequential   16228 ms    source.jsearch 15248 · source.adzuna 979
```

Two things are visible here that a single total would hide.

**Sum versus max.** Sequential spends 15248 + 979 = 16227 ms; concurrent spends the slowest
source alone. That is the entire argument for the fan-out, in two span trees.

**The quota trade-off, made concrete.** Sequential has *two* source spans; concurrent has
three. Adzuna returned enough jobs that the cascade never needed Remotive — so sequential
never asked it, while concurrent had already spent the request. Same results either way, but
one of them costs an extra API call. `SCOUT_CONCURRENT_SOURCES=false` buys the quota back
and pays in latency, which is exactly the choice `docs/optimizing_latency.md` describes.

And in both modes: 15 seconds of JSearch, `sources_used = ['adzuna']`. Hold that number —
it is the bug the rest of this notebook hands to Ollie.

## 6. Capability 1 — trace investigation

Open the project in Opik, open one of the two traces above, and open the Ollie panel.

1. **"This search took N seconds. Which part was slow?"**
   With per-source spans it can name `source.jsearch`. Without them the only honest answer
   is the total you already knew.

2. **"Did the slow source contribute any results?"**
   The answer is in `sources_used` on the trace output. The slow thing was also the useless
   thing, and no amount of staring at the total would have said so.

This is the capability the notebook can prepare for you. The next three need Ollie to reach
your code, which is a decision only you can make.

## 7. Capabilities 2-4 — the codebase loop

These need a local bridge, which grants Ollie read access to this repository and the ability
to propose writes to it (each one approved by you) and to run your agent. Start it in its own
terminal, from the repo root:

```bash
uv run opik connect --project job-scout
```

and stop it when you are done:

```bash
uv run opik connect stop --project job-scout
```

Nothing in this notebook starts it for you — that is deliberate. It is a real grant of access
to a directory on your machine and it should be a decision, not a cell you ran past.

Then, in the Ollie panel:

**2. Source-code integration** — *"Read the code behind the source.jsearch span and tell me
where that 15 seconds comes from."* It should land on `JSearchSource.__init__` in
`src/job_scout/tools/jobs_api.py` and the `timeout: float = 15.0` default. **Open the file
yourself.** An assistant that names the right file and the wrong line is more dangerous than
one that says it does not know, and this is the cheap moment to find out.

**3. Propose a fix** — *"Propose a change that stops one slow source holding up the whole
search. Keep the cascade's consumption order and thresholds unchanged."* Judge the answer
against [`../docs/optimizing_latency.md`](../docs/optimizing_latency.md), which records what
was already tried and one honest failure. Approve the write only after reading `git diff`.

**4. Rerun and verify** — *"Rerun that search with the original inputs"*, then *"Run the
job-scout-search-suite against the updated agent."* The suite's before-number is **33%**
(1/3), red on exactly the two assertions the bug violates. A fix that works moves it to 100%;
a fix that only looks right will not. That is why the gate exists before the fix does.

The cell below shows you the suite's assertions and the current baseline without spending
anything.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from setup_search_suite import ASSERTIONS, CASES, SLOW_SOURCE_MS, SUITE_NAME  # noqa: E402

print(f"{SUITE_NAME}: {len(CASES)} cases, slow-source threshold {SLOW_SOURCE_MS} ms\n")
for i, assertion in enumerate(ASSERTIONS, 1):
    print(f"{i}. {assertion}\n")

print("baseline measured 2026-08-05 — red on the two the timeout violates:")
print("   no source over 8s          33%  (1/3)")
print("   every used source helped   67%  (2/3)")
print("   search returned jobs      100%  (3/3)")
print("   pass rate                  33%")
print("\nRe-run it yourself (~$0.04):  uv run python scripts/setup_search_suite.py --run --yes")

## 8. Capability 5 — cross-workspace search

Ollie queries traces, datasets, experiments and prompts in one conversation. Phase 2 and 3
left plenty to ask about, and the useful trick is to **ask it things you already know first**:

> "Compare the tailoring-gpt-4.1-mini experiments before and after the prompt optimization."

You know the answer — fabrication rate 0.309 → 0.1423, in
[`../docs/phase3_findings.md`](../docs/phase3_findings.md). If Ollie's number disagrees, you
have learned something important about the tool before trusting it on a question you *cannot*
check.

> "Show me the versions of the tailor prompt and what changed."

> "Which traces in this project have fabrication_flags above zero?"

> "What is in the job-scout-tailoring-cases dataset?"

### The honest close

Ollie did not find the 15-second timeout. The instrumentation did — because somebody decided
a job source deserved its own span — and Ollie read it out loud, then followed it into the
code. That is a genuinely useful thing for a tool to do, and it is not the same as the tool
doing your observability for you.

Every question in this notebook was answerable only because the trace already contained the
answer. The engineering is deciding what to record; the assistant is the interface to it.

Full click paths, the screenshot shot-list and the release checklist are in
[`../docs/ollie.md`](../docs/ollie.md).